# Pre-entrega 6 — Orquestador turístico multiagente

Este notebook reutiliza los módulos Python del repositorio público. Ejecutá las celdas en orden; no es necesario subir archivos del proyecto ni crear un `.env`.

**Requisito previo:** publicar la carpeta `Pre-entrega6/` completa, incluido este notebook, en la rama `main` de [curso-ai-engineering](https://github.com/mcolatto/curso-ai-engineering). La primera celda comprobará que los módulos y catálogos estén disponibles en el clon.

Usá un entorno Python 3.12 o posterior, con CPU: no se necesita GPU. La ejecución requiere acceso a Internet.


## 1. Clonar el repositorio y comprobar las rutas

Se descarga la rama `main`. Si ya existe el clon en esta sesión, se reutiliza sin hacer `pull` ni sobrescribir sus archivos. Para descargar una publicación más reciente, iniciá un entorno de ejecución nuevo y ejecutá nuevamente el notebook. Se muestra el commit utilizado para identificar la versión ejecutada.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/mcolatto/curso-ai-engineering.git"
REPO_DIR = Path("/content/curso-ai-engineering")

if sys.version_info < (3, 12):
    raise RuntimeError("Seleccioná un entorno con Python 3.12 o posterior.")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)

PROJECT_DIR = REPO_DIR / "Pre-entrega6"
required = [
    "requirements.txt", "config.py", "schemas.py", "state.py", "llm.py", "tools.py",
    "graph.py", "main.py", "demo_refinamiento.py", "agents/__init__.py",
    "agents/supervisor.py", "agents/hoteleria.py", "agents/turismo.py", "agents/analista.py",
    "data/hoteles.json", "data/turismo.json",
]
missing = [name for name in required if not (PROJECT_DIR / name).is_file()]
if missing:
    raise RuntimeError(
        "Pre-entrega6 todavía no está publicada completa en main. "
        "Publicá los archivos del proyecto antes de ejecutar desde Colab. Faltan: " + ", ".join(missing)
    )
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Directorio:", Path.cwd())
print("Commit:", commit)
print("Python:", sys.version.split()[0])


## 2. Instalar las dependencias del proyecto

Primero se instala `requirements.txt`. Luego, sólo en Colab, se instala `jedi` para IPython y se fija `langchain-core==1.7.4` para compatibilidad con el `langchain` preinstalado. El entorno local conserva las versiones de `requirements.txt`. Finalmente, `pip check` verifica las dependencias del entorno completo.

Esta celda descarga paquetes, pero no realiza solicitudes al modelo. Si Colab pide reiniciar la sesión para aplicar las versiones instaladas, reiniciala y volvé a ejecutar desde la primera celda.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
# Compatibilidad con los paquetes preinstalados de Colab.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jedi", "langchain-core==1.7.4"], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)


## 3. Configurar OpenAI mediante los secretos de Colab

En el panel lateral **Secretos** (ícono de llave), creá el secreto `OPENAI_API_KEY` y habilitá el acceso para este notebook. La celda lo lee con `google.colab.userdata` y lo mantiene en el entorno del proceso: no lo imprime ni lo escribe en archivos.

Podés editar los parámetros siguientes antes de ejecutar la celda. El modelo debe admitir herramientas y salida estructurada; el esfuerzo vacío omite ese parámetro. No se envía temperatura.


In [ ]:
from google.colab import userdata

try:
    clave = userdata.get("OPENAI_API_KEY")
    if not isinstance(clave, str) or not clave.strip():
        raise ValueError("Secreto vacío")
except Exception:
    raise RuntimeError(
        "No se pudo leer OPENAI_API_KEY. Creá el secreto y habilitá su acceso a este notebook."
    ) from None
os.environ["OPENAI_API_KEY"] = clave.strip()
del clave

os.environ.update({
    "OPENAI_MODEL": "gpt-6.1-sol",
    "OPENAI_TIMEOUT_SECONDS": "30",
    "OPENAI_MAX_TOKENS": "4096",
    "OPENAI_REASONING_EFFORT": "",
    "RECURSION_LIMIT": "10",
})
print("Configuración lista. La credencial no se muestra ni se guarda en archivos.")


## 4. Ejecutar los tres escenarios

Esta celda **consume API de OpenAI**. El recorrido previsto realiza unas 23 llamadas: siete por escenario normal y nueve en la prueba controlada si el Supervisor pide refinamiento. Puede variar según sus decisiones, hasta nueve por escenario dentro del flujo permitido. Repetir la celda vuelve a consumir API y utiliza threads nuevos.

`main.py` ejecuta el Supervisor, los investigadores y el Analista; muestra las decisiones y contribuciones, persiste checkpoints en SQLite y genera `traza_ejecucion.json`.

1. Febrero: respuesta completa.
2. Marzo: respuesta parcial por alojamiento incompatible con las fechas.
3. Prueba controlada: se omite únicamente la evaluación gastronómica del primer análisis. El Supervisor decide si solicita una corrección. Un feedback genérico, una omisión no detectada o una corrección insuficiente producen un resultado no satisfactorio, que también se conserva en la traza.

La verificación del feedback es textual y conservadora; su instrucción exacta queda disponible para revisión. La prueba no ordena al Supervisor refinar ni sustituye sus decisiones.

Colab permite `await` directamente: reutilizamos la función asíncrona existente sin llamar a `asyncio.run()` dentro de su event loop.


In [ ]:
from config import TRACE_PATH
from main import main as ejecutar_demostracion

traza_anterior = TRACE_PATH.read_bytes() if TRACE_PATH.exists() else None
codigo_salida = await ejecutar_demostracion()
traza_actual = TRACE_PATH.read_bytes() if TRACE_PATH.exists() else None
traza_generada_en_esta_ejecucion = traza_actual is not None and traza_actual != traza_anterior
print("Código de salida:", codigo_salida)
if not traza_generada_en_esta_ejecucion:
    print("No se generó una traza nueva. Una traza previa del repositorio no es evidencia de esta ejecución.")


## 5. Revisar la traza de esta ejecución

La consola anterior es el detalle de las transiciones. Este resumen lee el JSON generado, incluyendo el resultado de la prueba controlada y su feedback observado. Si hubo un fallo previo que impidió guardar una traza nueva, no se muestra la evidencia antigua como si fuera de esta ejecución.

SQLite vive en el entorno temporal de Colab. La traza se puede descargar con la celda siguiente antes de cerrar la sesión. No contiene la API key.


In [ ]:
import json

if traza_generada_en_esta_ejecucion:
    traza = json.loads(traza_actual)
    for nombre, caso in traza.items():
        print("\n===", nombre, "===")
        print("Nodos:", " → ".join(evento["nodo"] for evento in caso["eventos"]))
        estado = caso.get("estado_final", {})
        decision = estado.get("decision_supervisor") or {}
        print("Evaluación interna:", decision.get("evaluacion_supervisor"))
        print("Refinamientos:", estado.get("refinamientos_analista", 0))
        if "resultado_prueba" in caso:
            print(json.dumps(caso["resultado_prueba"], ensure_ascii=False, indent=2))
else:
    print("No hay una traza nueva para revisar.")


## 6. Descargar la evidencia (opcional)

Descarga la traza de esta ejecución, tanto si la prueba controlada fue satisfactoria como si falló. El código de la aplicación y los catálogos permanecen en el clon; no se suben archivos a GitHub desde este notebook.


In [ ]:
from google.colab import files

if traza_generada_en_esta_ejecucion:
    files.download(str(TRACE_PATH))
else:
    print("No se generó una traza nueva para descargar.")
